In [30]:
import socket
import networkx as nx
import sys,time
import numpy as np
import random
import matplotlib.cm as cmx
import math

from sdt_draw_utils import draw_sdt_nx,draw_sdt_nx_nodes,draw_sdt_nx_edges,draw_sdt_nx_node_labels,draw_sdt_nx_edge_labels,sdt_nx_circular,sdt_nx_random,sdt_nx_spectral,sdt_nx_spring,sdt_nx_shell,set_status,pan

In [31]:
try:
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
except socket.error as msg:
    print("[ERROR] %s\n" % msg)
    sys.exit(1)
# 
try:
    sock.connect(("127.0.0.1", 50000))
except socket.error as msg:
    sys.stderr.write("[ERROR] %s\n" % msg)
    sys.exit(2)

In [32]:
# Get some networkx graph
#clear things    
sock.sendall('clear all '.encode())
#    sock.sendall('clear nodes ')    
sock.sendall('layer Worldwind,off '.encode())
sock.sendall('layer Sdt::Kml,off '.encode())
sock.sendall('layer "All Layers::Sdt::Node Labels,off" '.encode())
sock.sendall('backgroundColor black '.encode())
sock.sendall('origin 0.0,0.0,0.0 '.encode())
sock.sendall('center 0.0,0,0.0,0.0,c '.encode())
#    sock.sendall('flyto 0.005,0.005,3000.0 ')
sock.sendall('flyto 0.005,0.005,3000.0 '.encode())
#sock.sendall(sdt_com)
#    sock.sendall("follow all,on ") 
#    draw_sdt_nx(sock,G,node_size=20,alpha=0.7,edge_color="blue",width=0.3)
#Try a colormap
sock.sendall('title "A Protean Christmas" '.encode())
sock.sendall('showSdtStatusPanel,off '.encode())

In [33]:
colormap = cmx.get_cmap("rainbow")
numnodes=500
colors = []
for i in range(numnodes):
    colors.append(random.uniform(0.0,1.0))
#Setup tree or cone surface
height = 0.8
radius = 0.4
tangent = radius/height
sl_height = math.sqrt(height*height+radius*radius)
pos = []

for i in range(numnodes):
#generate a random 3d position on a cone
# Steps: generate random height then random cirle position given height.
#
#        rh=random.betavariate(2,4)
#        rh=np.random.lognormal(mean=0.3,sigma=0.2)
    placed_on_tree = False
    while not placed_on_tree:
        rh= np.random.uniform(0,1)
        rh= rh * height
        top=height-rh
        rradius = top*tangent
    # Need more prob on placing in relationship to radius
        prob = rradius/radius
        if np.random.uniform(0,1) < prob:
            placed_on_tree = True
    radians = random.uniform(0.0,math.pi*2.0*1.0)
    x = (math.sin(radians)*rradius)
    y = (math.cos(radians)*rradius)
    pos.append((x,y,rh))
nodes = [x for x in range(numnodes)]
G = nx.Graph()
G.add_nodes_from(nodes)
draw_sdt_nx_nodes(sock,G,pos,geoPos=False,
            node_color=colors,
            node_shape="sphere",
            node_size=20,
            n_alpha=0.7,
            cmap=colormap)
H=nx.Graph()
H.add_node(0) #Node was previously named star
starpos=[(0.0,0.0,height+0.05)]
draw_sdt_nx_nodes(sock,H,starpos,geoPos=False,
            node_color="yellow",
            node_shape="Sphere",
            node_size=60,
            n_alpha=0.9)

In [34]:
sock.sendall('flyto 0.000,0.00,4000.0 '.encode())

In [35]:
sock.sendall('clear links '.encode())
G.remove_edges_from(G.edges())
# make some edges
limit = 0.07
for n,n_pos in enumerate(pos):
    for k, k_pos in enumerate(pos):
        dist = np.sqrt((k_pos[0]-n_pos[0])**2.0 
                           + (k_pos[1]-n_pos[1])**2.0 
                           + (k_pos[2]-n_pos[2])**2.0)
        if n != k and dist < limit:
            G.add_edge(n,k)
draw_sdt_nx_edges(sock,G,pos,geoPos=False,
            edge_color="white",
            line_widths=1.0,
            l_alpha=0.5,
            style='solid')
sock.sendall('collapseLinks on '.encode())

In [36]:
on = True
while True:
    if on:
        sock.sendall('layer "All Layers::Sdt::Network Links,off" '.encode())
        on=False
    else:
        sock.sendall('layer "All Layers::Sdt::Network Links,on" '.encode())
        on=True        
    time.sleep(random.uniform(0.5,3.0))

BrokenPipeError: [Errno 32] Broken pipe